In [12]:
import os
import pandas as pd
from src.influx.connection import get_influx_client

In [13]:
PLAN_PATH = "data/exports/extraction_plan.csv"

In [14]:
def read_csv_flexible(path: str, nrows: int | None = None) -> pd.DataFrame:
    """
    Lê CSV tentando vírgula e ponto-e-vírgula.
    Permite limitar a quantidade de linhas lidas com nrows.
    """
    df = pd.read_csv(path, nrows=nrows)

    if "measurement_id" in df.columns:
        return df

    df = pd.read_csv(path, sep=";", nrows=nrows)

    if "measurement_id" in df.columns:
        return df

    raise ValueError(
        f"Nao foi possivel identificar as colunas esperadas no arquivo {path}. "
        f"Colunas encontradas: {list(df.columns)}"
    )

In [15]:
def build_device_tag(device_id: str) -> str:
    """
    Monta o valor da tag deviceId no formato esperado pelo Influx.
    Se já vier em formato URL, mantém.
    """
    device_id = str(device_id).strip()

    if device_id.startswith("https://react2020.eu/device/"):
        return device_id

    return f"https://react2020.eu/device/{device_id}"

In [16]:
def get_time_bounds_from_plan(plan_df: pd.DataFrame, field: str = "value") -> list[dict]:
    #client = get_influx_client()
    rows = []

    # remove duplicidade para não consultar a mesma combinação várias vezes
    plan_df = (
        plan_df[["measurement_id", "device_id", "unit"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    for _, row in plan_df.iterrows():
        measurement = str(row["measurement_id"]).strip()
        device_id_raw = str(row["device_id"]).strip()
        device_tag = build_device_tag(device_id_raw)
        print(device_tag)
        # unit = "" if pd.isna(row["unit"]) else str(row["unit"]).strip()

        # query_min = (
        #     f'SELECT first("{field}") AS first_val '
        #     f'FROM "{measurement}" '
        #     f'WHERE "deviceId" = \'{device_tag}\''
        # )

        # query_max = (
        #     f'SELECT last("{field}") AS last_val '
        #     f'FROM "{measurement}" '
        #     f'WHERE "deviceId" = \'{device_tag}\''
        # )

        # print(f"[RUN] measurement={measurement} / deviceId={device_tag}")

        # result_min = client.query(query_min)
        # result_max = client.query(query_max)

        # points_min = list(result_min.get_points())
        # points_max = list(result_max.get_points())

        # if not points_min or not points_max:
        #     print(f"[INFO] Nenhum dado para {measurement} / {device_tag}")
        #     rows.append({
        #         "measurement_id": measurement,
        #         "device_id": device_id_raw,
        #         "device_tag": device_tag,
        #         "unit": unit,
        #         "time_min": None,
        #         "time_max": None
        #     })
        #     continue

        # time_min = points_min[0].get("time")
        # time_max = points_max[0].get("time")

        # rows.append({
        #     "measurement_id": measurement,
        #     "device_id": device_id_raw,
        #     "device_tag": device_tag,
        #     "unit": unit,
        #     "time_min": time_min,
        #     "time_max": time_max
        # })

    return rows

In [17]:
def export_time_bounds():
    plan_df = read_csv_flexible(PLAN_PATH, nrows=3)
    bounds = get_time_bounds_from_plan(plan_df, field="value")
    df = pd.DataFrame(plan_df)
    return df 

In [18]:
df = export_time_bounds()
print(df)

https://react2020.eu/device/SMA-3008628290-EDMM
https://react2020.eu/device/SMA-3008628305-EDMM
https://react2020.eu/device/VIC-GXHQ20503L3SA-41
   active measurement_id             device_id unit              time_min  \
0    True      eAcGridIn   SMA-3008628290-EDMM   Wh  2026-03-02T00:00:00Z   
1    True      eAcGridIn   SMA-3008628305-EDMM   Wh  2026-03-02T00:00:00Z   
2    True      eAcGridIn  VIC-GXHQ20503L3SA-41   Wh  2026-03-02T00:00:00Z   

               time_max output_format  
0  2026-03-06T00:00:00Z           csv  
1  2026-03-06T00:00:00Z           csv  
2  2026-03-06T00:00:00Z           csv  
